# 🏅 Olympic Winter Games — AI Chatbot

This notebook lets you query the Olympic Winter Games SQLite database using **natural language**.
It uses **Claude AI** (Anthropic) to translate your questions into SQL and execute them automatically.

> Make sure you have run `pipeline.py run` first so that `data/olympics.db` exists.

## 1. Imports & Setup

In [ ]:
import os
import sqlite3
import pandas as pd
import anthropic
from dotenv import load_dotenv

load_dotenv()  # Loads ANTHROPIC_API_KEY from .env

DB_PATH = "../data/olympics.db"

client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

print("Setup complete.")

## 2. Helper: Fetch Database Schema

In [ ]:
def get_schema(db_path: str = DB_PATH) -> str:
    """
    Returns a human-readable schema string for all tables in the SQLite database.
    This is passed to Claude so it understands the data structure.
    """
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Get all table names
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [row[0] for row in cursor.fetchall()]

    schema_parts = []
    for table in tables:
        cursor.execute(f"PRAGMA table_info({table});")
        columns = cursor.fetchall()
        col_defs = ", ".join(f"{col[1]} ({col[2]})" for col in columns)
        schema_parts.append(f"Table '{table}': {col_defs}")

    conn.close()
    return "\n".join(schema_parts)


print(get_schema())

## 3. Helper: Generate SQL with Claude

In [ ]:
def generate_sql(question: str, schema: str) -> str:
    """
    Sends the user's question + database schema to Claude.
    Returns the generated SQL query as a plain string.
    """
    system_prompt = (
        "You are an expert SQLite data analyst. "
        "You will be given a database schema and a user question. "
        "Respond with ONLY a valid SQLite SQL query — no explanation, no markdown, no backticks. "
        "The query must be directly executable."
    )

    user_prompt = f"""Database schema:
{schema}

Question: {question}

SQL query:"""

    response = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=512,
        system=system_prompt,
        messages=[
            {"role": "user", "content": user_prompt}
        ]
    )

    sql = response.content[0].text.strip()
    return sql

## 4. Helper: Execute SQL

In [ ]:
def run_sql(sql: str, db_path: str = DB_PATH) -> pd.DataFrame:
    """
    Executes a SQL query against the SQLite database.
    Returns the result as a pandas DataFrame.
    """
    conn = sqlite3.connect(db_path)
    try:
        df = pd.read_sql_query(sql, conn)
    finally:
        conn.close()
    return df

## 5. Main `ask()` Function^

In [ ]:
def ask(question: str, verbose: bool = False) -> pd.DataFrame:
    """
    Ask any natural language question about the Olympic Winter Games.
    
    Args:
        question: A question in plain English.
        verbose:  If True, also prints the generated SQL query.
    
    Returns:
        A pandas DataFrame with the query results.
    """
    schema = get_schema()
    sql = generate_sql(question, schema)

    if verbose:
        print(f"📝 Generated SQL:\n{sql}\n")

    result = run_sql(sql)
    return result

print("✅ ask() is ready. Start querying below!")